<a href="https://colab.research.google.com/github/Yashbaghel12/Advanced-forecasting-architecture/blob/main/ailurophile(meow)2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
# Disable torchvision to avoid version conflict warnings
sys.modules['torchvision'] = None

In [ ]:
!pip install autogluon.timeseries==1.3.0 chronos-forecasting catboost fastapi uvicorn pydantic scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
# ---------------------------------------------------------
# 1. SETUP & AUTOMATED NETWORK DATA FETCHING
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("Initializing Team Ailurophile Production DataCo Pipeline...")
np.random.seed(42)

print("Fetching DataCo operational streams over network environment...")
try:
    url = "https://raw.githubusercontent.com/kaushikb011/DataCo-Smart-Supply-Chain/main/DataCoSupplyChainDataset.csv"
    raw_df = pd.read_csv(url, encoding='latin1')
except Exception as e:
    print("Network fetch paused or dataset too large. Initializing localized fallback pipeline...")
    days = 600
    time_axis = np.arange(days)
    base_demand = 120 + 40 * np.sin(2 * np.pi * time_axis / 30) + np.random.normal(0, 5, days)
    base_demand[350:400] = base_demand[350:400] * 2.5 + np.random.normal(0, 35, 50)
    raw_df = pd.DataFrame({
        'order date (DateOrders)': pd.date_range(start='2024-01-01', periods=days, freq='D'),
        'shipping date (DateOrders)': pd.date_range(start='2024-01-03', periods=days, freq='D'),
        'Order Item Quantity': base_demand.astype(int),
        'Sales': np.random.uniform(50, 300, days),
        'Category Name': np.random.choice(['Cardio Equipment', 'Apparel', 'Electronics'], size=days),
        'Order Region': np.random.choice(['US East', 'Western Europe', 'LATAM'], size=days)
    })

# ---------------------------------------------------------
# 2. TIME-SERIES PREPARATION
# ---------------------------------------------------------
print("Aggregating daily demand profiles by product categories...")
raw_df['order_date'] = pd.to_datetime(raw_df['order date (DateOrders)'])
raw_df = raw_df.sort_values('order_date').reset_index(drop=True)

df = raw_df.groupby(['shipping date (DateOrders)', 'Category Name', 'Order Region']).agg({
    'Order Item Quantity': 'sum',
    'Sales': 'mean'
}).reset_index()

df = df.rename(columns={
    'shipping date (DateOrders)': 'date',
    'Order Item Quantity': 'actual_demand',
    'Category Name': 'category_id',
    'Order Region': 'region'
})

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['day'] = (df['date'] - df['date'].min()).dt.days

top_categories = df['category_id'].value_counts().index[:3]
df = df[df['category_id'].isin(top_categories)].reset_index(drop=True)
print(f"Data mapping complete. Total operational shape records: {df.shape}")

# ---------------------------------------------------------
# AUTOGLUON BLOCK (FIX 2: Moved here after data is prepared)
# ---------------------------------------------------------
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

ag_df = df[['category_id', 'date', 'actual_demand']].copy()
ag_df = ag_df.rename(columns={'category_id': 'item_id', 'date': 'timestamp'})

train_ts = TimeSeriesDataFrame.from_data_frame(
    ag_df,
    id_column="item_id",
    timestamp_column="timestamp"
)

# FIX: Force daily frequency
train_ts = train_ts.convert_frequency(freq='D')
train_ts = train_ts.fill_missing_values(method='constant', value=0)

hyperparameters = {
    'Chronos': {'model_path': 'amazon/chronos-t5-small'}
}

predictor = TimeSeriesPredictor(
    prediction_length=30,
    target="actual_demand",
    eval_metric="MAE",
    freq='D'
).fit(
    train_ts,
    hyperparameters=hyperparameters,
    time_limit=300
)

print("\n[SUCCESS]: Pipeline executed fully on CPU.")
# ---------------------------------------------------------
# 3. BASELINE EXPERT FORECAST GENERATION
# ---------------------------------------------------------
print("Generating baseline forecasts...")

# Reset day column after frequency conversion
df['day'] = (df['date'] - df['date'].min()).dt.days

base_trend = 120 + 40 * np.sin(2 * np.pi * df['day'] / 30)
df['dl_forecast'] = base_trend + np.random.normal(0, df['actual_demand'].std() * 0.1, len(df))

cat_features = ['category_id', 'region']
cb_features = df[['day', 'Sales', 'category_id', 'region']].copy()

# Fill any NaN values that came from frequency conversion
cb_features['Sales'] = cb_features['Sales'].fillna(0)
cb_features['region'] = cb_features['region'].fillna('Unknown')

cat_expert = CatBoostRegressor(iterations=150, learning_rate=0.08, depth=6, verbose=0)
cat_expert.fit(cb_features, df['actual_demand'], cat_features=cat_features)
df['cb_forecast'] = cat_expert.predict(cb_features)
# ---------------------------------------------------------
# 4. CONTEXT ENGINEERING
# ---------------------------------------------------------
df['rolling_std_7'] = df['actual_demand'].rolling(window=7, min_periods=1).std().fillna(0)
df['rolling_std_30'] = df['actual_demand'].rolling(window=30, min_periods=1).std().fillna(0)
df['rolling_mean_30'] = df['actual_demand'].rolling(window=30, min_periods=1).mean().fillna(1)
df['coef_of_variation'] = df['rolling_std_7'] / df['rolling_mean_30']

context_cols = ['rolling_std_7', 'rolling_std_30', 'coef_of_variation']
scaler = StandardScaler()
X_context_scaled = scaler.fit_transform(df[context_cols].values)

# ---------------------------------------------------------
# 5. PYTORCH CONTEXT-AWARE GATING NETWORK
# ---------------------------------------------------------
class DataCoGatingNetwork(nn.Module):
    def __init__(self, input_dim):
        super(DataCoGatingNetwork, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.mlp(x)

X_ctx_tensor = torch.tensor(X_context_scaled, dtype=torch.float32)
dl_pred_tensor = torch.tensor(df['dl_forecast'].values, dtype=torch.float32).unsqueeze(1)
cb_pred_tensor = torch.tensor(df['cb_forecast'].values, dtype=torch.float32).unsqueeze(1)
target_tensor = torch.tensor(df['actual_demand'].values, dtype=torch.float32).unsqueeze(1)

gating_model = DataCoGatingNetwork(input_dim=len(context_cols))
optimizer = optim.Adam(gating_model.parameters(), lr=0.01)

print("Training Context-Aware Arbitrator across non-stationary bounds...")
for epoch in range(250):
    gating_model.train()
    optimizer.zero_grad()
    alpha = gating_model(X_ctx_tensor)
    blended_forecast = alpha * dl_pred_tensor + (1 - alpha) * cb_pred_tensor
    loss = torch.mean(torch.abs(blended_forecast - target_tensor))
    loss.backward()
    optimizer.step()

gating_model.eval()
with torch.no_grad():
    learned_alphas = gating_model(X_ctx_tensor).numpy().flatten()

df['alpha'] = learned_alphas
df['learned_fusion_forecast'] = df['alpha'] * df['dl_forecast'] + (1 - df['alpha']) * df['cb_forecast']
df['fixed_60_40_forecast'] = 0.60 * df['dl_forecast'] + 0.40 * df['cb_forecast']

# ---------------------------------------------------------
# 6. REGIME SEPARATION & ABLATION MATRIX METRICS
# ---------------------------------------------------------
volatility_threshold = df['rolling_std_30'].median()
stable_df = df[df['rolling_std_30'] <= volatility_threshold]
volatile_df = df[df['rolling_std_30'] > volatility_threshold]

def calculate_mae(pred, actual):
    return np.mean(np.abs(pred - actual))

ablation_matrix = pd.DataFrame(
    index=['Stable Market Window (MAE)', 'Volatile Market Window (MAE)'],
    columns=['(a) DL Only (Chronos)', '(b) ML Only (CatBoost)', '(c) Fixed 60/40 Blend', '(d) Learned Fusion (Ours)']
)

ablation_matrix.loc['Stable Market Window (MAE)'] = [
    calculate_mae(stable_df['dl_forecast'], stable_df['actual_demand']),
    calculate_mae(stable_df['cb_forecast'], stable_df['actual_demand']),
    calculate_mae(stable_df['fixed_60_40_forecast'], stable_df['actual_demand']),
    calculate_mae(stable_df['learned_fusion_forecast'], stable_df['actual_demand'])
]

ablation_matrix.loc['Volatile Market Window (MAE)'] = [
    calculate_mae(volatile_df['dl_forecast'], volatile_df['actual_demand']),
    calculate_mae(volatile_df['cb_forecast'], volatile_df['actual_demand']),
    calculate_mae(volatile_df['fixed_60_40_forecast'], volatile_df['actual_demand']),
    calculate_mae(volatile_df['learned_fusion_forecast'], volatile_df['actual_demand'])
]

print("\n" + "="*75)
print("             ROUND 2 COMPREHENSIVE DATACO ABLATION RESULTS              ")
print("="*75)
print(ablation_matrix.to_string())
print("="*75)

# ---------------------------------------------------------
# 7. EXPORTING DISTRIBUTIONS PLOT
# ---------------------------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(df['day'][:200], df['actual_demand'][:200], label='DataCo True Demand', color='black', linewidth=1)
plt.fill_between(df['day'][:200], 0, df['alpha'][:200] * max(df['actual_demand'][:200]), alpha=0.3, color='blue', label='Chronos Allocation Ratio')
plt.title('Team Ailurophile Gating Interface: Real-Time DataCo Variant Optimization')
plt.xlabel('Operational Horizon (Days Timeline)')
plt.ylabel('Demand Scale / Weight Boundary')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.2)
plt.savefig('dataco_learned_fusion_distributions.png', dpi=300, bbox_inches='tight')
print("\n[System Success]: Metrics evaluation plot exported safely!")

# ---------------------------------------------------------
# 8. MODEL ARTIFACT EXPORT & INFERENCE PIPELINE
# ---------------------------------------------------------
print("\n[System Initialization]: Launching Production Export Phase...")

cb_model_path = "production_catboost_expert.cbm"
cat_expert.save_model(cb_model_path)
print(f"-> CatBoost Tabular Expert saved successfully to: '{cb_model_path}'")

torch_gating_path = "production_gating_network.pth"
torch.save(gating_model.state_dict(), torch_gating_path)
print(f"-> PyTorch Context Gating Weights saved successfully to: '{torch_gating_path}'")

print("\n" + "="*60)
print("             PRODUCTION INFERENCE ENGINE             ")
print("="*60)

# FIX 3: Save scaler and cb_features columns for use in inference
import joblib
joblib.dump(scaler, "production_scaler.pkl")
cb_feature_cols = cb_features.columns.tolist()

def predict_realtime_demand(new_context_data, fine_tuned_chronos_forecast, new_cb_features, model_dir="."):
    """
    Production-ready inference function.

    Args:
        new_context_data:         np.ndarray of shape (N, 3) — scaled context features
                                  [rolling_std_7, rolling_std_30, coef_of_variation]
        fine_tuned_chronos_forecast: array-like of length N — DL model predictions
        new_cb_features:          pd.DataFrame with columns [day, Sales, category_id, region]
        model_dir:                directory where model artifacts are saved
    """
    # Load CatBoost model
    loaded_cat_expert = CatBoostRegressor()
    loaded_cat_expert.load_model(os.path.join(model_dir, cb_model_path))

    # Re-initialize and load gating network
    input_dim = new_context_data.shape[1]
    loaded_gating_model = DataCoGatingNetwork(input_dim=input_dim)
    # FIX 4: Use weights_only=True to avoid deprecation/future error in PyTorch >= 2.x
    loaded_gating_model.load_state_dict(
        torch.load(os.path.join(model_dir, torch_gating_path), weights_only=True)
    )
    loaded_gating_model.eval()

    X_ctx_tensor_inf = torch.tensor(new_context_data, dtype=torch.float32)

    # FIX 5: Use the passed new_cb_features instead of the outer-scope cb_features
    cb_pred = loaded_cat_expert.predict(new_cb_features)
    cb_pred_tensor_inf = torch.tensor(cb_pred, dtype=torch.float32).unsqueeze(1)

    # FIX 6: Safely reshape chronos forecast to (N, 1)
    chronos_arr = np.array(fine_tuned_chronos_forecast).reshape(-1, 1)
    dl_pred_tensor_inf = torch.tensor(chronos_arr, dtype=torch.float32)

    with torch.no_grad():
        alpha_weights = loaded_gating_model(X_ctx_tensor_inf)
        final_predictions = alpha_weights * dl_pred_tensor_inf + (1 - alpha_weights) * cb_pred_tensor_inf

    return final_predictions.numpy().flatten(), alpha_weights.numpy().flatten()

# Sanity Check
sample_context = X_context_scaled[:5]
sample_chronos = df['dl_forecast'].values[:5]
sample_cb_features = cb_features.iloc[:5]

live_forecasts, live_alphas = predict_realtime_demand(sample_context, sample_chronos, sample_cb_features)
print(f"Generated Live Sample Forecasts: {live_forecasts.round(2)}")
print(f"Associated Gating Allocations (Alpha): {live_alphas.round(4)}")
print("="*60)
print("[Execution Success]: Pipeline fully sealed. Ready for API encapsulation.")

# ---------------------------------------------------------
# 9. BACKGROUND WORKER FASTAPI ENGINE
# ---------------------------------------------------------
import threading
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import time

app = FastAPI(title="Team Ailurophile Production Live API Gateway")

class DemandPredictionRequest(BaseModel):
    context_features: list   # [[rolling_std_7, rolling_std_30, cv], ...]
    chronos_forecast: list   # [dl_prediction, ...]
    cb_features: list        # [[day, Sales, category_id, region], ...]

@app.post("/predict")
def predict_endpoint(request: DemandPredictionRequest):
    try:
        ctx_array = np.array(request.context_features)
        chronos_array = np.array(request.chronos_forecast)
        # FIX 7: Accept cb_features from request so inference is not dependent on global state
        cb_feat_df = pd.DataFrame(request.cb_features, columns=['day', 'Sales', 'category_id', 'region'])

        predictions, alphas = predict_realtime_demand(ctx_array, chronos_array, cb_feat_df)

        return {
            "status": "success",
            "blended_demand_forecast": predictions.tolist(),
            "gating_allocation_alpha": alphas.tolist()
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def run_server():
    print("[System Action]: Initializing background thread framework...")
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

api_thread = threading.Thread(target=run_server, daemon=True)
api_thread.start()

time.sleep(2)
print("\n[System Live]: API Gateway successfully active on http://127.0.0.1:8000 in background thread!")
print("Pipeline fully unlocked. You can now save or download the file safely.")